# 04 - Temporal Feature Engineering

## Objetivo

Construir o dataset temporal de modelagem em nivel `user_window_id-product_id` para o sistema de recomendacao de proximo carrinho.

Este notebook consome os candidatos temporais gerados no notebook `03-candidate-strategy.ipynb` e adiciona features calculadas somente com eventos historicos cujo `order_number <= history_end_order_number` da janela.

## Inputs

- `data/processed/orders_product_unified.parquet`
- `data/features/temporal_candidates_v1/`
- `data/raw/products.csv`

## Output

- `data/features/temporal_modeling_dataset_v1/`

## Unidade de modelagem

A unidade final deixa de ser apenas `user_id-product_id` e passa a ser `user_window_id-product_id`.

Isso e necessario porque o mesmo usuario aparece em diferentes janelas temporais. Um mesmo par `user_id-product_id` pode ter valores diferentes de historico dependendo do limite da janela.

## Controles de leakage

- `target` e usado apenas como rotulo supervisionado.
- Nenhuma feature usa produtos do `target_order_id`.
- Cada janela e processada com corte temporal proprio.
- Features historicas usam somente eventos com `order_number <= history_end_order_number`.
- O notebook preserva metadados de janela para auditoria: `split`, `user_window_id`, `target_order_id`, `target_order_number` e `history_end_order_number`.

## Continuidade com o notebook anterior

As familias de features de usuario, usuario-produto e categoria foram mantidas no mesmo espirito do notebook `04-feature-engineering.ipynb` anterior, mas agora calculadas no limite temporal de cada janela.

Features temporalizadas equivalentes ao padrao anterior:

- `user_prior_order_count`
- `user_avg_basket_size`
- `user_avg_days_between_orders`
- `user_reorder_rate`
- `user_has_single_prior_order`
- `user_product_purchase_count`
- `user_product_reorder_count`
- `user_product_avg_add_to_cart_order`
- `user_product_was_bought_before`
- `user_product_orders_since_last_purchase`
- `user_product_days_since_last_purchase`
- `aisle_id`
- `department_id`
- `user_aisle_purchase_count`
- `user_department_purchase_count`

## Features removidas por leakage temporal

As features globais de produto do notebook anterior nao foram mantidas nesta versao:

- `product_prior_purchase_count`
- `product_prior_reorder_rate`
- `product_popularity_pct`
- `product_popularity_log`

No notebook anterior elas eram calculadas sobre todo o conjunto `prior`. Na formulacao temporal, isso faria janelas antigas enxergarem compras futuras do proprio periodo `prior`, criando leakage temporal. Recalcular popularidade global por janela seria possivel, mas exigiria uma linha do tempo global e aumentaria bastante o custo do notebook. Por isso, nesta versao o sinal de popularidade global fica restrito a `candidate_source` e `candidate_rank`, que ja vem da etapa de geracao de candidatos e tem o trade-off documentado no notebook 03.

## Novas features e valor preditivo esperado

- `candidate_source`: indica se o candidato veio de recompra, similaridade, categoria ou global. Valor preditivo: produtos de recompra tendem a ter maior probabilidade de compra; similaridade e categoria ajudam a separar descoberta personalizada de fallback global.
- `candidate_rank`: ordem em que o produto foi selecionado pela estrategia de candidatos. Valor preditivo: candidatos mais altos geralmente vieram de sinais mais fortes ou mais prioritarios.
- `history_order_count`: quantidade de pedidos historicos disponiveis na janela. Valor preditivo: mede maturidade do usuario; usuarios com mais historico tendem a ter padroes de recompra mais estaveis.
- `history_unique_products`: diversidade de produtos ja comprados na janela. Valor preditivo: diferencia usuarios com historico concentrado de usuarios exploratorios.
- `history_group`: segmento derivado de `history_unique_products`. Valor preditivo: permite ao modelo aprender comportamentos diferentes para usuarios P0-P50, P50-P90 e P90+.
- `user_total_items`: total de itens comprados no historico da janela. Valor preditivo: aproxima intensidade de consumo e complementa `user_prior_order_count` e `user_avg_basket_size`.
- `user_product_purchase_share`: participacao do produto no historico da janela do usuario. Valor preditivo: normaliza a frequencia do produto pelo tamanho do historico; ajuda a comparar usuarios com volumes muito diferentes.

Colunas como `split`, `window_number`, `user_window_id`, `target_order_id`, `target_order_number`, `history_start_order_number` e `history_end_order_number` sao mantidas para auditoria, particionamento e avaliacao. Elas nao devem ser usadas diretamente como features do modelo de ranking sem uma justificativa explicita.

## Imputacoes estruturais

As imputacoes seguem o padrao do notebook anterior, adaptadas para o limite temporal da janela:

- produto nunca comprado pelo usuario recebe `user_product_purchase_count = 0`, `user_product_reorder_count = 0`, `user_product_was_bought_before = 0` e `user_product_avg_add_to_cart_order = 0`.
- `user_product_orders_since_last_purchase` ausente recebe `user_prior_order_count` da janela.
- `user_product_days_since_last_purchase` ausente recebe `user_avg_days_between_orders * user_prior_order_count` da janela.
- contagens de aisle e department ausentes recebem `0`.


---

## 1. Setup inicial


In [1]:
import gc
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)


In [2]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

UNIFIED_DATASET_PATH = PROCESSED_DIR / "orders_product_unified.parquet"
PRODUCTS_PATH = RAW_DIR / "products.csv"
TEMPORAL_CANDIDATES_DIR = FEATURES_DIR / "temporal_candidates_v1"
TEMPORAL_MODELING_DATASET_DIR = FEATURES_DIR / "temporal_modeling_dataset_v1"

OVERWRITE_TEMPORAL_MODELING_DATASET = True
PRELOAD_PRIOR_IN_MEMORY = True
PROGRESS_EVERY_PARTITIONS = 5

for input_path in [UNIFIED_DATASET_PATH, PRODUCTS_PATH, TEMPORAL_CANDIDATES_DIR]:
    assert input_path.exists(), f"Entrada nao encontrada: {input_path}"

candidate_part_paths = sorted(TEMPORAL_CANDIDATES_DIR.glob("*.parquet"))
assert candidate_part_paths, f"Nenhuma particao encontrada em: {TEMPORAL_CANDIDATES_DIR}"

print("Entradas encontradas com sucesso.")
print(f"Particoes de candidatos: {len(candidate_part_paths):,}")
print(f"Output temporal sera salvo em: {TEMPORAL_MODELING_DATASET_DIR}")


Entradas encontradas com sucesso.
Particoes de candidatos: 47
Output temporal sera salvo em: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/temporal_modeling_dataset_v1


### 1.1 Schema dos candidatos


In [3]:
candidate_schema = pq.ParquetFile(candidate_part_paths[0]).schema.names

required_candidate_columns = {
    "user_id",
    "product_id",
    "split",
    "window_number",
    "user_window_id",
    "target_order_id",
    "target_order_number",
    "history_start_order_number",
    "history_end_order_number",
    "history_order_count",
    "history_unique_products",
    "history_group",
    "candidate_source",
    "candidate_rank",
    "target",
}

missing_candidate_columns = sorted(required_candidate_columns - set(candidate_schema))
assert not missing_candidate_columns, (
    f"Colunas ausentes nos candidatos temporais: {missing_candidate_columns}"
)

print(f"Colunas nos candidatos: {len(candidate_schema):,}")
print(candidate_schema)


Colunas nos candidatos: 15
['user_id', 'product_id', 'split', 'window_number', 'user_window_id', 'target_order_id', 'target_order_number', 'history_start_order_number', 'history_end_order_number', 'history_order_count', 'history_unique_products', 'history_group', 'candidate_source', 'candidate_rank', 'target']


In [4]:
product_catalog = pd.read_csv(
    PRODUCTS_PATH,
    usecols=["product_id", "aisle_id", "department_id"],
)

assert product_catalog["product_id"].is_unique, (
    "Catalogo de produtos contem product_id duplicado."
)

print(f"Produtos no catalogo: {len(product_catalog):,}")


Produtos no catalogo: 49,688


---

## 2. Funcoes de leitura temporal


In [5]:
prior_columns = [
    "order_id",
    "product_id",
    "add_to_cart_order",
    "reordered",
    "user_id",
    "eval_set",
    "order_number",
    "days_since_prior_order",
]

unified_dataset = ds.dataset(UNIFIED_DATASET_PATH, format="parquet")
prior_by_user = None


def load_prior_for_users(user_ids):
    user_ids = [int(user_id) for user_id in user_ids]

    if prior_by_user is not None:
        prior_parts = [
            prior_by_user[user_id]
            for user_id in user_ids
            if user_id in prior_by_user
        ]
        if not prior_parts:
            return pd.DataFrame(columns=prior_columns)

        return pd.concat(prior_parts, ignore_index=True)

    user_filter = ds.field("user_id").isin(user_ids)
    prior_filter = ds.field("eval_set") == "prior"

    table = unified_dataset.to_table(
        columns=prior_columns,
        filter=prior_filter & user_filter,
    )

    prior_df = table.to_pandas()

    if prior_df.empty:
        return prior_df

    prior_df = prior_df.sort_values(
        ["user_id", "order_number", "add_to_cart_order"],
        kind="mergesort",
    ).reset_index(drop=True)

    return prior_df


### 1.2 Carregamento rapido do historico prior

Para evitar escanear o Parquet unificado uma vez por particao, o modo rapido carrega uma unica fotografia do `prior` dos usuarios elegiveis com apenas as colunas necessarias. Em maquinas com pouca memoria, defina `PRELOAD_PRIOR_IN_MEMORY = False`.


In [6]:
eligible_user_ids = set()

for candidate_part_path in candidate_part_paths:
    part_users = pd.read_parquet(candidate_part_path, columns=["user_id"])
    eligible_user_ids.update(map(int, part_users["user_id"].unique()))

print(f"Usuarios elegiveis nos candidatos temporais: {len(eligible_user_ids):,}")


Usuarios elegiveis nos candidatos temporais: 115,909


In [7]:
if PRELOAD_PRIOR_IN_MEMORY:
    prior_filter = ds.field("eval_set") == "prior"
    user_filter = ds.field("user_id").isin(list(eligible_user_ids))

    prior_table = unified_dataset.to_table(
        columns=prior_columns,
        filter=prior_filter & user_filter,
    )
    prior_all_df = prior_table.to_pandas()

    prior_all_df = prior_all_df.sort_values(
        ["user_id", "order_number", "add_to_cart_order"],
        kind="mergesort",
    ).reset_index(drop=True)

    prior_by_user = {
        int(user_id): user_prior.copy()
        for user_id, user_prior in prior_all_df.groupby("user_id", sort=False)
    }

    print(f"Prior carregado em memoria: {len(prior_all_df):,} linhas")
    print(f"Usuarios com historico prior: {len(prior_by_user):,}")
else:
    print("Modo streaming ativo: o prior sera filtrado por particao.")


Prior carregado em memoria: 20,205,983 linhas
Usuarios com historico prior: 115,909


### 1.3 Amostra de uma particao


In [8]:
sample_candidates = pd.read_parquet(candidate_part_paths[0]).head(10)

sample_candidates

,user_id,product_id,split,window_number,user_window_id,target_order_id,target_order_number,history_start_order_number,history_end_order_number,history_order_count,history_unique_products,history_group,candidate_source,candidate_rank,target
0,1,196,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,1,1
1,1,12427,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,2,1
2,1,10258,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,3,1
3,1,25133,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,4,1
4,1,13032,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,5,0
5,1,13176,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,6,0
6,1,26405,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,7,0
7,1,26088,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,8,0
8,1,10326,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,9,0
9,1,17122,train,1,1_train_3108588,3108588,8,1,7,7,13,P0-P50,recompra,10,0


In [9]:
def build_order_timeline(history_df):
    order_timeline = (
        history_df
        .groupby("order_number", as_index=False)
        .agg(
            order_id=("order_id", "first"),
            basket_size=("product_id", "count"),
            days_since_prior_order=("days_since_prior_order", "first"),
        )
        .sort_values("order_number")
        .reset_index(drop=True)
    )

    order_timeline["days_since_prior_order"] = (
        order_timeline["days_since_prior_order"].fillna(0)
    )
    order_timeline["cum_days"] = order_timeline["days_since_prior_order"].cumsum()

    return order_timeline


In [10]:
def summarize_window(history_df, user_window_id, history_end_order_number):
    order_timeline = build_order_timeline(history_df)
    end_cum_days = float(order_timeline["cum_days"].iloc[-1])

    user_summary = {
        "user_window_id": user_window_id,
        "user_prior_order_count": int(order_timeline["order_id"].nunique()),
        "user_avg_basket_size": float(order_timeline["basket_size"].mean()),
        "user_avg_days_between_orders": float(
            order_timeline["days_since_prior_order"].mean()
        ),
        "user_reorder_rate": float(history_df["reordered"].mean()),
        "user_total_items": int(len(history_df)),
    }


    user_summary["user_has_single_prior_order"] = int(
        user_summary["user_prior_order_count"] <= 1
    )

    product_summary = (
        history_df
        .groupby("product_id", as_index=False)
        .agg(
            user_product_purchase_count=("product_id", "size"),
            user_product_reorder_count=("reordered", "sum"),
            user_product_avg_add_to_cart_order=("add_to_cart_order", "mean"),
            user_product_last_order_number=("order_number", "max"),
        )
    )

    cum_days_by_order = order_timeline.set_index("order_number")["cum_days"]
    product_summary["user_product_orders_since_last_purchase"] = (
        history_end_order_number - product_summary["user_product_last_order_number"]
    )
    product_summary["user_product_days_since_last_purchase"] = (
        end_cum_days
        - product_summary["user_product_last_order_number"].map(cum_days_by_order)
    )
    product_summary["user_product_was_bought_before"] = 1
    product_summary["user_product_purchase_share"] = (
        product_summary["user_product_purchase_count"] / len(history_df)
    )

    aisle_summary = (
        history_df
        .merge(product_catalog, on="product_id", how="left")
        .groupby("aisle_id", as_index=False)
        .agg(user_aisle_purchase_count=("product_id", "size"))
    )

    department_summary = (
        history_df
        .merge(product_catalog, on="product_id", how="left")
        .groupby("department_id", as_index=False)
        .agg(user_department_purchase_count=("product_id", "size"))
    )

    return user_summary, product_summary, aisle_summary, department_summary


---

## 3. Inventario das features temporais

Esta secao consolida a decisao de feature engineering antes da materializacao do dataset.

### 3.1 Features historicas mantidas

As features abaixo preservam o mesmo significado do dataset anterior, mas passam a ser calculadas por janela:

- Usuario: `user_prior_order_count`, `user_avg_basket_size`, `user_avg_days_between_orders`, `user_reorder_rate`, `user_has_single_prior_order`, `user_total_items`.
- Usuario-produto: `user_product_purchase_count`, `user_product_reorder_count`, `user_product_avg_add_to_cart_order`, `user_product_was_bought_before`, `user_product_orders_since_last_purchase`, `user_product_days_since_last_purchase`, `user_product_purchase_share`.
- Categoria: `aisle_id`, `department_id`, `user_aisle_purchase_count`, `user_department_purchase_count`.

### 3.2 Features de candidatos adicionadas

`candidate_source` e `candidate_rank` foram preservadas porque carregam informacao sobre a forca da etapa de geracao de candidatos. Elas nao usam o `target` da janela, mas refletem a prioridade dos sinais de recompra, similaridade, categoria e popularidade global.

Essas features tendem a ser fortes em baselines e modelos de ranking porque indicam o quao naturalmente um produto entrou no conjunto de candidatos antes de qualquer modelo supervisionado.

### 3.3 Metadados temporais

`split`, `window_number`, `user_window_id`, `target_order_id`, `target_order_number`, `history_start_order_number` e `history_end_order_number` sao colunas de controle.

Elas permitem auditar leakage, agrupar ranking por janela e separar treino, validacao e teste. Por padrao, nao devem entrar como features numericas do modelo.

### 3.4 Features globais de produto removidas

As features globais de produto do notebook anterior foram removidas para evitar que janelas antigas usem popularidade observada em compras futuras do `prior`.

Essa remocao reduz um sinal preditivo conhecido, mas deixa o protocolo temporal mais limpo para comparar baselines e MLP.


---

## 4. Construção incremental das features


In [11]:
def build_features_for_candidates(candidates_df, prior_df):
    window_columns = [
        "user_window_id",
        "user_id",
        "history_end_order_number",
    ]
    window_meta = candidates_df[window_columns].drop_duplicates()

    history_df = prior_df.merge(
        window_meta,
        on="user_id",
        how="inner",
    )
    history_df = history_df[
        history_df["order_number"] <= history_df["history_end_order_number"]
    ].copy()

    assert not history_df.empty, "Historico temporal vazio na particao."

    order_timeline = (
        history_df
        .groupby(["user_window_id", "order_number"], as_index=False)
        .agg(
            order_id=("order_id", "first"),
            basket_size=("product_id", "count"),
            days_since_prior_order=("days_since_prior_order", "first"),
        )
        .sort_values(["user_window_id", "order_number"], kind="mergesort")
        .reset_index(drop=True)
    )
    order_timeline["days_since_prior_order"] = (
        order_timeline["days_since_prior_order"].fillna(0)
    )
    order_timeline["cum_days"] = (
        order_timeline
        .groupby("user_window_id")["days_since_prior_order"]
        .cumsum()
    )

    user_order_features = (
        order_timeline
        .groupby("user_window_id", as_index=False)
        .agg(
            user_prior_order_count=("order_id", "nunique"),
            user_avg_basket_size=("basket_size", "mean"),
            user_avg_days_between_orders=("days_since_prior_order", "mean"),
            user_window_end_cum_days=("cum_days", "max"),
        )
    )

    user_reorder_features = (
        history_df
        .groupby("user_window_id", as_index=False)
        .agg(
            user_reorder_rate=("reordered", "mean"),
            user_total_items=("product_id", "size"),
        )
    )

    user_features = user_order_features.merge(
        user_reorder_features,
        on="user_window_id",
        how="inner",
    )
    user_features["user_has_single_prior_order"] = (
        user_features["user_prior_order_count"].le(1).astype(int)
    )

    last_order_cum_days = order_timeline[
        ["user_window_id", "order_number", "cum_days"]
    ].rename(
        columns={
            "order_number": "user_product_last_order_number",
            "cum_days": "user_product_last_cum_days",
        }
    )

    product_summary = (
        history_df
        .groupby(["user_window_id", "product_id"], as_index=False)
        .agg(
            user_product_purchase_count=("product_id", "size"),
            user_product_reorder_count=("reordered", "sum"),
            user_product_avg_add_to_cart_order=("add_to_cart_order", "mean"),
            user_product_last_order_number=("order_number", "max"),
        )
    )
    product_summary = product_summary.merge(
        last_order_cum_days,
        on=["user_window_id", "user_product_last_order_number"],
        how="left",
    )
    product_summary = product_summary.merge(
        window_meta[["user_window_id", "history_end_order_number"]],
        on="user_window_id",
        how="left",
    )
    product_summary = product_summary.merge(
        user_features[["user_window_id", "user_window_end_cum_days", "user_total_items"]],
        on="user_window_id",
        how="left",
    )
    product_summary["user_product_orders_since_last_purchase"] = (
        product_summary["history_end_order_number"]
        - product_summary["user_product_last_order_number"]
    )
    product_summary["user_product_days_since_last_purchase"] = (
        product_summary["user_window_end_cum_days"]
        - product_summary["user_product_last_cum_days"]
    )
    product_summary["user_product_was_bought_before"] = 1
    product_summary["user_product_purchase_share"] = (
        product_summary["user_product_purchase_count"]
        / product_summary["user_total_items"]
    )
    product_summary = product_summary.drop(
        columns=[
            "user_product_last_order_number",
            "user_product_last_cum_days",
            "history_end_order_number",
            "user_window_end_cum_days",
            "user_total_items",
        ]
    )

    history_with_catalog = history_df.merge(
        product_catalog,
        on="product_id",
        how="left",
    )

    aisle_summary = (
        history_with_catalog
        .groupby(["user_window_id", "aisle_id"], as_index=False)
        .agg(user_aisle_purchase_count=("product_id", "size"))
    )

    department_summary = (
        history_with_catalog
        .groupby(["user_window_id", "department_id"], as_index=False)
        .agg(user_department_purchase_count=("product_id", "size"))
    )

    user_features = user_features.drop(columns=["user_window_end_cum_days"])

    features_df = candidates_df.merge(
        product_catalog,
        on="product_id",
        how="left",
    )
    features_df = features_df.merge(
        user_features,
        on="user_window_id",
        how="left",
    )
    features_df = features_df.merge(
        product_summary,
        on=["user_window_id", "product_id"],
        how="left",
    )
    features_df = features_df.merge(
        aisle_summary,
        on=["user_window_id", "aisle_id"],
        how="left",
    )
    features_df = features_df.merge(
        department_summary,
        on=["user_window_id", "department_id"],
        how="left",
    )

    assert features_df["user_prior_order_count"].notna().all(), (
        "Features de usuario ausentes apos merge temporal."
    )

    del history_df, order_timeline, history_with_catalog
    gc.collect()

    return features_df


In [12]:
def apply_structural_imputations(features_df):
    features_df["user_product_days_since_last_purchase"] = (
        features_df["user_product_days_since_last_purchase"]
        .fillna(
            features_df["user_avg_days_between_orders"]
            * features_df["user_prior_order_count"]
        )
    )

    features_df["user_product_orders_since_last_purchase"] = (
        features_df["user_product_orders_since_last_purchase"]
        .fillna(features_df["user_prior_order_count"])
    )

    zero_fill_columns = [
        "user_product_purchase_count",
        "user_product_reorder_count",
        "user_product_was_bought_before",
        "user_product_purchase_share",
        "user_aisle_purchase_count",
        "user_department_purchase_count",
        "user_product_avg_add_to_cart_order",
    ]

    for column in zero_fill_columns:
        features_df[column] = features_df[column].fillna(0)

    numeric_int_columns = [
        "user_product_purchase_count",
        "user_product_reorder_count",
        "user_product_orders_since_last_purchase",
        "user_product_was_bought_before",
        "user_aisle_purchase_count",
        "user_department_purchase_count",
        "user_has_single_prior_order",
    ]

    for column in numeric_int_columns:
        features_df[column] = features_df[column].astype("int64")

    return features_df


In [13]:
def validate_temporal_features(features_df, candidates_df, part_name):
    assert len(features_df) == len(candidates_df), (
        f"Numero de linhas mudou na particao {part_name}."
    )
    assert not features_df.duplicated(subset=["user_window_id", "product_id"]).any(), (
        f"Pares user_window_id-product_id duplicados em {part_name}."
    )
    assert features_df["target"].isin([0, 1]).all(), (
        f"Target invalido em {part_name}."
    )
    assert (
        features_df["history_end_order_number"]
        < features_df["target_order_number"]
    ).all(), f"Janela com historico posterior ao alvo em {part_name}."
    assert features_df["aisle_id"].notna().all(), (
        f"aisle_id ausente apos merge de catalogo em {part_name}."
    )
    assert features_df["department_id"].notna().all(), (
        f"department_id ausente apos merge de catalogo em {part_name}."
    )


In [14]:
if TEMPORAL_MODELING_DATASET_DIR.exists():
    assert OVERWRITE_TEMPORAL_MODELING_DATASET, (
        f"Diretorio ja existe: {TEMPORAL_MODELING_DATASET_DIR}. "
        "Defina OVERWRITE_TEMPORAL_MODELING_DATASET = True para recriar."
    )
    shutil.rmtree(TEMPORAL_MODELING_DATASET_DIR)

TEMPORAL_MODELING_DATASET_DIR.mkdir(parents=True, exist_ok=True)

partition_stats = []

for part_idx, candidate_part_path in enumerate(candidate_part_paths):
    partition_start = time.perf_counter()

    candidates_df = pd.read_parquet(candidate_part_path)
    read_candidates_at = time.perf_counter()

    users_in_part = candidates_df["user_id"].unique()
    prior_df = load_prior_for_users(users_in_part)
    load_prior_at = time.perf_counter()

    features_df = build_features_for_candidates(candidates_df, prior_df)
    build_features_at = time.perf_counter()

    features_df = apply_structural_imputations(features_df)

    validate_temporal_features(
        features_df=features_df,
        candidates_df=candidates_df,
        part_name=candidate_part_path.name,
    )
    validate_at = time.perf_counter()

    output_path = TEMPORAL_MODELING_DATASET_DIR / candidate_part_path.name
    features_df.to_parquet(output_path, index=False)
    write_at = time.perf_counter()

    partition_stats.append(
        {
            "partition": output_path.name,
            "rows": len(features_df),
            "users": features_df["user_id"].nunique(),
            "windows": features_df["user_window_id"].nunique(),
            "positives": int(features_df["target"].sum()),
            "nulls": int(features_df.isna().sum().sum()),
            "read_candidates_seconds": read_candidates_at - partition_start,
            "load_prior_seconds": load_prior_at - read_candidates_at,
            "build_features_seconds": build_features_at - load_prior_at,
            "validate_seconds": validate_at - build_features_at,
            "write_seconds": write_at - validate_at,
            "total_seconds": write_at - partition_start,
        }
    )

    if (part_idx + 1) % PROGRESS_EVERY_PARTITIONS == 0 or part_idx == 0:
        print(
            f"Particoes processadas: {part_idx + 1}/{len(candidate_part_paths)} | "
            f"ultima={write_at - partition_start:.1f}s | "
            f"build={build_features_at - load_prior_at:.1f}s | "
            f"write={write_at - validate_at:.1f}s"
        )

    del candidates_df, prior_df, features_df
    gc.collect()

partition_stats_df = pd.DataFrame(partition_stats)

partition_stats_df.head()


Particoes processadas: 1/47 | ultima=5.5s | build=4.0s | write=1.0s
Particoes processadas: 5/47 | ultima=5.2s | build=3.8s | write=0.9s
Particoes processadas: 10/47 | ultima=5.2s | build=3.8s | write=0.9s
Particoes processadas: 15/47 | ultima=5.0s | build=3.6s | write=0.9s
Particoes processadas: 20/47 | ultima=5.4s | build=3.9s | write=1.0s
Particoes processadas: 25/47 | ultima=5.4s | build=3.9s | write=1.0s
Particoes processadas: 30/47 | ultima=5.2s | build=3.8s | write=1.0s
Particoes processadas: 35/47 | ultima=5.2s | build=3.8s | write=0.9s
Particoes processadas: 40/47 | ultima=5.3s | build=3.8s | write=1.0s
Particoes processadas: 45/47 | ultima=5.4s | build=3.9s | write=1.0s


,partition,rows,users,windows,positives,nulls,read_candidates_seconds,load_prior_seconds,build_features_seconds,validate_seconds,write_seconds,total_seconds
0,part-0000.parquet,1973600,2467,9868,72809,0,0.170869,0.073486,3.987238,0.291160,0.973395,5.496149
1,part-0001.parquet,1973600,2467,9868,71756,0,0.138456,0.070858,3.996468,0.286368,0.899239,5.391390
2,part-0002.parquet,1973600,2467,9868,74116,0,0.117838,0.058029,3.904406,0.283514,0.882781,5.246569
3,part-0003.parquet,1973600,2467,9868,74255,0,0.118361,0.058873,3.938473,0.287367,0.907185,5.310260
4,part-0004.parquet,1973600,2467,9868,73195,0,0.113658,0.061935,3.819059,0.288417,0.907662,5.190731


---

## 5. Validacoes do artefato temporal


In [15]:
assert not partition_stats_df.empty, "Nenhuma particao foi processada."
assert partition_stats_df["nulls"].sum() == 0, "Dataset final contem valores nulos."

final_part_paths = sorted(TEMPORAL_MODELING_DATASET_DIR.glob("*.parquet"))
assert len(final_part_paths) == len(candidate_part_paths), (
    "Quantidade de particoes finais difere dos candidatos."
)

print(f"Particoes finais: {len(final_part_paths):,}")
print(f"Linhas finais: {partition_stats_df['rows'].sum():,}")
print(f"Positivos finais: {partition_stats_df['positives'].sum():,}")


Particoes finais: 47
Linhas finais: 92,727,200
Positivos finais: 3,454,844


In [16]:
modeling_dataset = ds.dataset(TEMPORAL_MODELING_DATASET_DIR, format="parquet")
dataset_columns = modeling_dataset.schema.names

forbidden_feature_columns = {
    "eval_set",
    "order_id",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "product_name",
    "aisle",
    "department",
}

leaked_columns = sorted(forbidden_feature_columns.intersection(dataset_columns))
assert not leaked_columns, f"Colunas transacionais proibidas no dataset final: {leaked_columns}"

print(f"Colunas finais: {len(dataset_columns):,}")
print(dataset_columns)


Colunas finais: 32
['user_id', 'product_id', 'split', 'window_number', 'user_window_id', 'target_order_id', 'target_order_number', 'history_start_order_number', 'history_end_order_number', 'history_order_count', 'history_unique_products', 'history_group', 'candidate_source', 'candidate_rank', 'target', 'aisle_id', 'department_id', 'user_prior_order_count', 'user_avg_basket_size', 'user_avg_days_between_orders', 'user_reorder_rate', 'user_total_items', 'user_has_single_prior_order', 'user_product_purchase_count', 'user_product_reorder_count', 'user_product_avg_add_to_cart_order', 'user_product_orders_since_last_purchase', 'user_product_days_since_last_purchase', 'user_product_was_bought_before', 'user_product_purchase_share', 'user_aisle_purchase_count', 'user_department_purchase_count']


In [17]:
validation_stats = []

for part_path in final_part_paths:
    part_df = pd.read_parquet(
        part_path,
        columns=[
            "split",
            "user_id",
            "user_window_id",
            "product_id",
            "target",
            "target_order_number",
            "history_end_order_number",
        ],
    )

    assert not part_df.duplicated(subset=["user_window_id", "product_id"]).any(), (
        f"Duplicidade por janela em {part_path.name}."
    )
    assert (part_df["history_end_order_number"] < part_df["target_order_number"]).all(), (
        f"Leakage temporal em {part_path.name}."
    )

    validation_stats.append(
        {
            "partition": part_path.name,
            "rows": len(part_df),
            "splits": ",".join(sorted(part_df["split"].unique())),
            "windows": part_df["user_window_id"].nunique(),
            "positives": int(part_df["target"].sum()),
        }
    )

    del part_df

validation_stats_df = pd.DataFrame(validation_stats)

validation_stats_df.groupby("splits", as_index=False).agg(
    partitions=("partition", "count"),
    rows=("rows", "sum"),
    windows=("windows", "sum"),
    positives=("positives", "sum"),
)


,splits,partitions,rows,windows,positives
0,"test,train,validation",47,92727200,463636,3454844


---

## 6. Persistencia final


In [18]:
print("Feature engineering temporal concluida.")
print(f"Dataset final: {TEMPORAL_MODELING_DATASET_DIR}")
print("Unidade de ranking preservada: user_window_id-product_id")
print("Target preservado apenas como rotulo supervisionado.")


Feature engineering temporal concluida.
Dataset final: /Users/helio/Library/CloudStorage/OneDrive-Pessoal/Pos_FIAP/Tech_Challanges/Fase2-Big-Data-Architecture/mlp-market-recommender-system/data/features/temporal_modeling_dataset_v1
Unidade de ranking preservada: user_window_id-product_id
Target preservado apenas como rotulo supervisionado.
